# 07 — Self-RAG Workflow

**Self-RAG** adds a *reflection* loop on top of basic retrieval. After retrieving documents, it:

1. **Grades** each document for relevance to the query
2. If documents are irrelevant, **rewrites the query** and retrieves again
3. **Generates** an answer from relevant documents
4. Repeats until relevant documents are found or max iterations reached

This is implemented both as a standalone strategy (`SelfRAGStrategy`) and as a LangGraph workflow (`SelfRAGWorkflow`).

## 1. Imports

In [ ]:
import sys
sys.path.insert(0, "..")

try:
    from backend.adaptive_rag.strategies.self_rag_strategy import SelfRAGStrategy
    from backend.adaptive_rag.langgraph_workflows.self_rag_workflow import SelfRAGWorkflow
    print("✓ Imported SelfRAGStrategy, SelfRAGWorkflow")
except ImportError as e:
    print(f"⚠ Backend import: {e}")

## 2. The Self-Reflection Loop

```
        ┌─────────────────────────────┐
        │                             │
        ▼                             │
   ┌─────────┐    ┌──────────┐    ┌────┴──────┐
   │ Retrieve │───▶│  Reflect │───▶│ Rewrite   │
   └─────────┘    └──────────┘    │ Query     │
        │                         └───────────┘
        │ (all relevant)
        ▼
   ┌──────────┐
   │ Generate  │
   └──────────┘
```

## 3. Build Components for SelfRAGStrategy

In [ ]:
from langchain.embeddings import FakeEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document

fake_emb = FakeEmbeddings(size=768)
docs = [
    Document(page_content="The Eiffel Tower is located in Paris, France.",
             metadata={"source": "geo.txt"}),
    Document(page_content="Python was created by Guido van Rossum and first released in 1991.",
             metadata={"source": "history.txt"}),
    Document(page_content="The human brain contains approximately 86 billion neurons.",
             metadata={"source": "biology.txt"}),
]
vector_store = FAISS.from_documents(docs, fake_emb)

class MockRetriever:
    def __init__(self, vs):
        self.vs = vs
    async def retrieve(self, query, top_k=5):
        from backend.adaptive_rag.retrievers.vector_retriever import VectorRetriever
        return await VectorRetriever(self.vs).retrieve(query, top_k)

class MockLLM:
    async def generate_with_context(self, query, context):
        return f"Self-RAG answer for '{query[:60]}' based on provided context."
    async def generate(self, prompt):
        return f"Fallback answer for: {prompt[:80]}"

class MockRelevanceGrader:
    async def grade(self, doc, query):
        # Simple overlap heuristic
        common = len(set(doc.lower().split()) & set(query.lower().split()))
        is_rel = common > 2
        return {"is_relevant": is_rel, "score": 1.0 if is_rel else 0.0, "explanation": "keyword"}

class MockHallucinationGrader:
    async def grade(self, answer, context):
        return {"is_grounded": True, "score": 1.0, "explanation": "heuristic"}

print("✓ Components built")

## 4. SelfRAGStrategy Demo

In [ ]:
import asyncio

retriever = MockRetriever(vector_store)
llm = MockLLM()
rel_grader = MockRelevanceGrader()
hall_grader = MockHallucinationGrader()

strategy = SelfRAGStrategy(retriever, llm, rel_grader, hall_grader)

async def demo_self_rag(query):
    result = await strategy.execute(query)
    print(f"Query     : {result['query']}")
    print(f"Strategy  : {result['strategy']}")
    print(f"Answer    : {result['answer'][:200]}")
    print(f"Iterations: {result.get('iterations', 'N/A')}")
    print(f"Sources   : {len(result['sources'])}")

asyncio.run(demo_self_rag("Who created Python and when?"))

## 5. Iterative Retrieval in Action

With a query that doesn't match any document, Self-RAG rewrites and retries:

In [ ]:
async def demo_iteration():
    result = await strategy.execute("What is the speed of light?")
    print(f"Original : {result['query']}")
    print(f"Answer   : {result['answer'][:200]}")
    print(f"Sources  : {len(result['sources'])}")
    if result.get('iterations'):
        print(f"Took {result['iterations']} iteration(s)")

asyncio.run(demo_iteration())

## 6. SelfRAGWorkflow (LangGraph)

The `SelfRAGWorkflow` implements the same loop as a **state machine** using LangGraph, with:
- A `SelfRAGState` TypedDict for tracking query, documents, reflection, answer, iteration
- Conditional edges that decide whether to reflect or generate

```python
from backend.adaptive_rag.langgraph_workflows.self_rag_workflow import SelfRAGWorkflow
```

The LangGraph version is what you'd deploy in production — it's more controllable and observable.

> **Next:** [08 — Corrective RAG](./08_Corrective_RAG.ipynb) — retrieve, grade, correct with web fallback